# MARVIN terrain model: fine-tune SegFormer-B0 on AI4Mars, export ONNX
Runs on a free Colab GPU (Runtime > Change runtime type > T4). About 30-45 minutes for 2 epochs.
**Untested draft:** written without access to Hugging Face, so the first data cell prints the label values. If they differ from soil/bedrock/sand/big rock = 0/1/2/3 (255 = unlabelled), edit `CLASSES` below.
Output: `marswalk_terrain.onnx` + `model.txt` + metrics. Dataset: hassanjbara/AI4MARS (CC0). Base weights: nvidia/mit-b0 (check its licence on the model card).

In [ ]:
!pip -q install transformers datasets onnx onnxruntime onnxscript evaluate

In [ ]:
import torch,numpy as np,random,json
from datasets import load_dataset
CLASSES=['soil','bedrock','sand','big_rock']   # label values 0..3; 255 = no label
ds=load_dataset('hassanjbara/AI4MARS')
print(ds)
ex=ds['train'][0];print(ex.keys());print(np.unique(np.array(ex['label_mask'])),np.array(ex['image']).shape)

In [ ]:
SIZE=384
MEAN=np.array([0.485,0.456,0.406],np.float32);STD=np.array([0.229,0.224,0.225],np.float32)
from PIL import Image
def prep(ex):
    im=ex['image'].convert('RGB').resize((SIZE,SIZE),Image.BILINEAR)
    lb=ex['label_mask'].resize((SIZE,SIZE),Image.NEAREST)
    x=((np.asarray(im,np.float32)/255-MEAN)/STD).transpose(2,0,1)
    y=np.asarray(lb).astype(np.int64)
    if y.ndim==3:y=y[...,0]
    y=np.where(y<len(CLASSES),y,255)
    return torch.from_numpy(x),torch.from_numpy(y)
class DS(torch.utils.data.Dataset):
    def __init__(s,d):s.d=d
    def __len__(s):return len(s.d)
    def __getitem__(s,i):return prep(s.d[i])
def labelled(d):   # some rows have no label mask (has_labels False): keep only labelled ones
    return d.select([i for i,h in enumerate(d['has_labels']) if h])
tr_l,te_l=labelled(ds['train']),labelled(ds['test_min3'])
print('labelled rows: train',len(tr_l),'of',len(ds['train']),'| test',len(te_l),'of',len(ds['test_min3']))
train=DS(tr_l);test=DS(te_l)
dl=torch.utils.data.DataLoader(train,batch_size=16,shuffle=True,num_workers=2,drop_last=True)
tl=torch.utils.data.DataLoader(test,batch_size=16,num_workers=2)

In [ ]:
from transformers import SegformerForSemanticSegmentation
dev='cuda'
model=SegformerForSemanticSegmentation.from_pretrained('nvidia/mit-b0',num_labels=len(CLASSES),id2label=dict(enumerate(CLASSES)),label2id={c:i for i,c in enumerate(CLASSES)}).to(dev)
opt=torch.optim.AdamW(model.parameters(),lr=6e-5,weight_decay=.01)
EPOCHS=2;steps=EPOCHS*len(dl);sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=2e-4,total_steps=steps,pct_start=.1)
scaler=torch.cuda.amp.GradScaler()
for ep in range(EPOCHS):
    model.train();tot=0
    for k,(x,y) in enumerate(dl):
        x,y=x.to(dev),y.to(dev)
        with torch.autocast('cuda',dtype=torch.float16):
            out=model(pixel_values=x,labels=y)   # loss ignores 255
        opt.zero_grad();scaler.scale(out.loss).backward();scaler.step(opt);scaler.update();sched.step();tot+=out.loss.item()
        if k%100==0:print(ep,k,round(out.loss.item(),3))
    print('epoch',ep,'mean loss',tot/len(dl))

In [ ]:
# Held-out evaluation on test_min3 (labels the annotators agreed on). Report this number honestly.
import torch.nn.functional as F
model.eval();conf=np.zeros((4,4),np.int64)
with torch.no_grad():
    for x,y in tl:
        lg=model(pixel_values=x.to(dev)).logits;lg=F.interpolate(lg,size=y.shape[-2:],mode='bilinear',align_corners=False)
        p=lg.argmax(1).cpu().numpy();y=y.numpy();m=y!=255
        conf+=np.bincount(y[m]*4+p[m],minlength=16).reshape(4,4)
iou=np.diag(conf)/(conf.sum(0)+conf.sum(1)-np.diag(conf)+1e-9)
res={'per_class_iou':dict(zip(CLASSES,iou.round(3).tolist())),'mIoU':float(iou.mean().round(3)),'pixel_accuracy':float((np.diag(conf).sum()/conf.sum()).round(3)),'test_split':'test_min3','train_epochs':EPOCHS,'input_size':SIZE}
print(res);json.dump(res,open('metrics.json','w'),indent=1)

In [ ]:
# Export to ONNX (input 1x3xSIZExSIZE, output logits 1x4xSIZE/4xSIZE/4)
class Wrap(torch.nn.Module):
    def __init__(s,m):super().__init__();s.m=m
    def forward(s,x):return s.m(pixel_values=x).logits
model.cpu().eval()
torch.onnx.export(Wrap(model),torch.zeros(1,3,SIZE,SIZE),'marswalk_terrain.onnx',input_names=['image'],output_names=['logits'],opset_version=17,dynamic_axes=None)
import onnxruntime as ort,os
s=ort.InferenceSession('marswalk_terrain.onnx');o=s.run(None,{'image':np.zeros((1,3,SIZE,SIZE),np.float32)})[0];print('onnx ok',o.shape,round(os.path.getsize('marswalk_terrain.onnx')/1e6,1),'MB')
open('model.txt','w').write(f'''input: {SIZE}x{SIZE} RGB, float32 NCHW, scaled 0-1 then normalised with mean {MEAN.tolist()} std {STD.tolist()}
output: logits 1x{len(CLASSES)}x{o.shape[2]}x{o.shape[3]} (upsample to image size, argmax)
classes in index order: {CLASSES}
source: SegFormer-B0 (nvidia/mit-b0 encoder) fine-tuned by us on AI4Mars (hassanjbara/AI4MARS, CC0), {EPOCHS} epochs
metrics: {json.dumps(res)}
licence: base weights nvidia/mit-b0, see its model card
''')

In [ ]:
# Download the files (Colab). Also save 3 qualitative examples from the test set for the slides.
from google.colab import files
for f in ['marswalk_terrain.onnx','model.txt','metrics.json']:files.download(f)